# 下一步:工程化与可解释性进阶

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/california_housing_nextsteps.ipynb)

本 notebook 是加州房价系列的**第三篇**(也是最后一篇)。前两篇:
- [`california_housing_tutorial.ipynb`](california_housing_tutorial.ipynb):走完一遍机器学习流程
- [`california_housing_advanced.ipynb`](california_housing_advanced.ipynb):5 个工程技巧让模型变强

这一篇教你**真实工作中天天用的「下一层」工具**——把代码变得专业、把模型变得可解释。

> 🎯 **写给谁看**:已经能跑通基础流程,想知道工程师/Kaggle 高手平时多写了什么的同学。
> **依然不需要太多前置知识**——每一节都从「旧方式有什么痛点」讲起。

## 你将学到 4 件「写出更专业代码 + 更可信结论」的工具

| # | 工具 | 它解决了什么 |
| --- | --- | --- |
| 1 | **`Pipeline`** | 数据预处理 + 模型 = 一个对象。**杜绝数据泄漏** |
| 2 | **`ColumnTransformer`** | 对不同列做不同预处理(数值标准化、类别独热) |
| 3 | **`cross_validate`** | 一次拿到 RMSE / MAE / R² 多个指标,而不是只一个 |
| 4 | **`HistGradientBoostingRegressor`** | sklearn 自带的「LightGBM 替代品」,免依赖、快 5–10 倍 |
| 5 | **SHAP**(可解释性) | 「这个样本被预测成 X,具体是哪些特征推动的?」 |

最后我们用一节做**导读**,告诉你想再深入应该看什么书、关注哪些前沿话题、哪里能找到更多数据。

## 目录
0. 阅读须知
1. 加载数据(快速复用上一份的)
2. **`Pipeline`**:把预处理 + 模型打包成一个对象
3. **`ColumnTransformer`**:不同列不同处理
4. **`cross_validate`**:一次拿到多个评估指标
5. **`HistGradientBoostingRegressor`**:sklearn 原生快速 GBM
6. **SHAP 可解释性入门**:为什么模型这样预测?
7. 进一步学习导读(ISLR / 前沿 DL / 数据源)
8. 总结


## 0. 阅读须知

> 📘 **小知识:为什么要在意「工程化」?**
> 训练模型只是机器学习项目的 10%,**剩下 90% 是把模型可靠地集成到生产系统**。
> 工程化工具(如 Pipeline)让你的代码:
> 1. **更难写错**(杜绝训练/测试用了不同 scaler 这种 bug);
> 2. **更易部署**(一个对象 → 一个文件 → 一行代码加载);
> 3. **更易复现**(超参搜索能直接搜索整个 pipeline)。

本 notebook 的所有代码都基于上一份用过的**加州房价数据**,保持连贯。

依赖(Kaggle 默认环境全部内置):
- `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`
- `shap` ← 第 6 节才用到。如果本地没有:`!pip install shap`


## 1. 环境与数据加载


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

bunch = fetch_california_housing(as_frame=True)
df = bunch.frame
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'训练: {X_train.shape}, 测试: {X_test.shape}')


## 2. `Pipeline`:把预处理 + 模型打包成一个对象

### 2.1 旧方式的痛点

下面这段代码看起来没问题,但**藏着一个非常常见的 bug**——找一找:

```python
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = Ridge()
model.fit(X_train_s, y_train)

# 然后做 5 折交叉验证
cross_val_score(model, X_train_s, y_train, cv=5)  # ❌ 数据泄漏!
```

> ❗ **找到 bug 了吗?**
> `X_train_s` 是用**整个训练集**的均值方差标准化的。
> 但交叉验证的每一折都把训练集再切成「子训练集 + 子验证集」——
> 子验证集里的均值方差**已经被 `scaler.fit` 看到了**。
> 这意味着 5 折 CV 的成绩会**比真实泛化好**,你被自己骗了。

正确做法:**每一折独立 fit scaler**。但手写很繁琐——这就是 `Pipeline` 要解决的事。

### 2.2 Pipeline 怎么用

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge()),
])
```

它把多个步骤串成一个对象。当你调用 `pipe.fit(X, y)`:
1. 先 `scaler.fit_transform(X)`;
2. 再把结果喂给 `model.fit(...)`。

**关键好处**:`pipe` 整体进 `cross_val_score`,每一折会自动**只在该折的训练子集上 fit 整个 pipeline**——
**数据泄漏自动解决**。

### 2.3 代码对比


In [ ]:
# 用 Pipeline 封装:scaler + 模型一气呵成
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

# 这一行同时做了:fit_transform(scaler) + fit(model)
pipe.fit(X_train, y_train)

# 预测时也是一气呵成:transform(scaler) + predict(model)
pred = pipe.predict(X_test)
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.4f}')
print(f'Test R²:   {r2_score(y_test, pred):.4f}')

# Pipeline 也能直接喂给 cross_val_score / GridSearchCV
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error')
print(f'5-fold CV RMSE: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}')


> 📘 **小知识:Pipeline 还能怎么用?**
> - `pipe.named_steps['scaler']` → 拿到底层的 scaler 对象;
> - `pipe['model'].coef_` → 直接访问模型的系数;
> - 配合 `GridSearchCV`,参数名写成 `'model__alpha'`(注意双下划线),可以同时调多个步骤。


## 3. `ColumnTransformer`:不同列不同处理

### 3.1 真实数据通常是「混合类型」

加州房价全是数值列,所以前面我们一个 `StandardScaler` 就搞定。但真实场景里:

- **数值列**:年龄、收入、面积 → 标准化或归一化;
- **类别列**:城市、性别、产品类型 → 独热编码(One-Hot)或目标编码;
- **日期列**:订单时间 → 拆成 年/月/日/星期;
- **文本列**:用户评论 → TF-IDF 或词嵌入。

`ColumnTransformer` 让你**对不同列指定不同的处理器**,然后整体喂给 Pipeline。

### 3.2 演示:给加州数据制造一个「类别列」

为了演示,我们把 `HouseAge` 离散化成 `age_bucket`(三档:新房/中等/老房)模拟一个类别特征。


In [ ]:
# 制造一个类别特征
def make_age_bucket(age):
    return pd.cut(age, bins=[-1, 15, 35, 100],
                  labels=['new', 'mid', 'old']).astype(str)

X_train2 = X_train.assign(age_bucket=make_age_bucket(X_train['HouseAge']))
X_test2  = X_test.assign(age_bucket=make_age_bucket(X_test['HouseAge']))

# 留下原始 HouseAge 作为对比,也保留新做的 age_bucket
print(X_train2[['HouseAge', 'age_bucket']].head(5))
print('\nage_bucket 类别分布:')
print(X_train2['age_bucket'].value_counts())


### 3.3 用 ColumnTransformer 分别处理

我们告诉 sklearn:
- 数值列 → `StandardScaler`
- 类别列 `age_bucket` → `OneHotEncoder`(变成 3 个 0/1 列)


In [ ]:
numeric_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                'Population', 'AveOccup', 'Latitude', 'Longitude']
categorical_cols = ['age_bucket']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                      numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', Ridge(alpha=1.0)),
])

pipe.fit(X_train2, y_train)
pred = pipe.predict(X_test2)
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.4f}')

# 看一下 ColumnTransformer 输出了什么列
feature_names_out = pipe.named_steps['prep'].get_feature_names_out()
print(f'\n变换后的列名 (共 {len(feature_names_out)} 列):')
for n in feature_names_out: print(' ', n)


> 📘 **小知识:`handle_unknown='ignore'` 是什么?**
> 训练集里只见过 `new/mid/old` 三档,如果测试集突然出现 `super_old`,
> 没这个参数会直接报错。设成 `ignore` 就把没见过的类别全部独热为 0。
> **生产环境必备**——上线后总会遇到训练时没见过的类别。


## 4. `cross_validate`:一次拿到多个评估指标

### 4.1 旧方式

`cross_val_score` 只能要一个指标:

```python
cross_val_score(pipe, X, y, cv=5, scoring='neg_root_mean_squared_error')
```

想同时看 RMSE、MAE、R²?要跑 3 次。`cross_validate` **一次跑多个**。

### 4.2 用法


In [ ]:
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae':  'neg_mean_absolute_error',
    'r2':   'r2',
}

cv_results = cross_validate(
    pipe, X_train2, y_train,
    cv=5,
    scoring=scoring,
    return_train_score=True,   # 同时返回训练集分数,可以看过拟合
)

# cv_results 是个字典,每个键对应一个长度=cv 的数组
print('cv_results keys:', list(cv_results.keys()))
print()

# 把它整理成 DataFrame 看
df_cv = pd.DataFrame({
    'fold': range(1, 6),
    'train_RMSE': -cv_results['train_rmse'],
    'test_RMSE':  -cv_results['test_rmse'],
    'test_MAE':   -cv_results['test_mae'],
    'test_R²':    cv_results['test_r2'],
    'fit_time':   cv_results['fit_time'],
})
print(df_cv.round(4))
print()
print(f"训练 RMSE 均值: {-cv_results['train_rmse'].mean():.4f}")
print(f"测试 RMSE 均值: {-cv_results['test_rmse'].mean():.4f}  (差距越大越过拟合)")


> 📘 **小知识:`return_train_score=True` 为啥重要?**
> 如果训练 RMSE 和测试 RMSE 差不多,说明模型**欠拟合**(增加复杂度);
> 如果训练 RMSE 远小于测试 RMSE,说明**过拟合**(增加正则化或减少复杂度)。
> 这是判断「下一步该怎么改进」最直接的信号。


## 5. `HistGradientBoostingRegressor`:sklearn 原生快速 GBM

### 5.1 它是什么?

简称 `HistGBR`,sklearn ≥ 0.21 自带的**直方图梯度提升**实现——
基本就是把 LightGBM 的核心算法搬进 sklearn,**API 完全统一,免装第三方依赖**。

| 维度 | `GradientBoostingRegressor` | `HistGradientBoostingRegressor` | LightGBM |
| --- | --- | --- | --- |
| 出处 | sklearn 原生 | sklearn 原生 | 第三方库 |
| 算法 | 教科书 GBM | 直方图分箱 GBM | 直方图 + leaf-wise |
| 速度 | 慢(基线) | 中等(快 5–20×) | 最快(快 50–100×) |
| API | 标准 sklearn | 标准 sklearn | 自家 API + sklearn wrapper |
| **何时选?** | 小数据 / 教学 | **绝大多数场景的最佳默认** | 极致性能 / 大数据 |

> 💡 **建议**:在 sklearn 项目里,**不要再用 `GradientBoostingRegressor` 了**——
> 直接换 `HistGBR`,代码改一行,速度涨一档。

### 5.2 速度对比演示


In [ ]:
# 同样的超参,跑两个版本对比
gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
hist = HistGradientBoostingRegressor(
    max_iter=200, max_depth=4, learning_rate=0.1, random_state=42)

# 老牌
t0 = time.time()
gbr.fit(X_train, y_train)
gbr_time = time.time() - t0
gbr_rmse = np.sqrt(mean_squared_error(y_test, gbr.predict(X_test)))

# 直方图版
t0 = time.time()
hist.fit(X_train, y_train)
hist_time = time.time() - t0
hist_rmse = np.sqrt(mean_squared_error(y_test, hist.predict(X_test)))

print(f'{"模型":<32} {"训练时间":>10} {"测试 RMSE":>12}')
print(f'{"GradientBoostingRegressor":<32} {gbr_time:>9.2f}s {gbr_rmse:>12.4f}')
print(f'{"HistGradientBoostingRegressor":<32} {hist_time:>9.2f}s {hist_rmse:>12.4f}')
print(f'\n速度提升: {gbr_time / hist_time:.1f}×,精度: 不相上下')


**结论**:**默认参数** + **HistGBR**,你能用更少的代码、更短的时间,做出更稳的模型。

> 📘 **小知识:HistGBR 还有个隐藏功能**
> 它**原生支持类别特征**——在 `fit` 时给 `categorical_features=[index_list]` 即可,
> 不需要先做 OneHot。在大量类别特征的场景下,这能省下大量内存。


## 6. SHAP 可解释性入门

### 6.1 为什么需要 SHAP?

我们之前用 `feature_importances_` 看过特征重要性,但它只回答了**「整体最重要的是哪些特征?」**

真实工作中你常常被问:
- 「为什么这个**具体**样本的房价被预测成 \$280K?」
- 「同样收入水平,为什么海边的房子贵 \$50K?」
- 「我把 `HouseAge` 改成 30,房价预测会怎么变?」

`feature_importances_` 答不了这些。SHAP 可以。

### 6.2 SHAP 值是什么?(直觉)

> 📘 **核心思想——博弈论的 Shapley 值**
> 假设 8 个特征是 8 位「球员」,共同把房价从「所有人的平均值」推到「这个样本的预测值」。
> SHAP 值告诉你:**每位球员各贡献了多少「推力」**。
> 推力为正 → 把价格推高;为负 → 把价格压低。

数学保证:**所有特征的 SHAP 值之和 = 该样本的预测值 − 所有训练样本的平均预测值**。
这是 SHAP 比其它「特征重要性」方法都更可靠的根本原因——它是**有数学定义的归因**,不是启发式。


### 6.3 训练一个模型 + 算 SHAP 值


In [ ]:
import shap

# 用 HistGBR,SHAP 对树模型有专门的高效实现 TreeExplainer
model = HistGradientBoostingRegressor(
    max_iter=200, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
print(f'测试 RMSE: {np.sqrt(mean_squared_error(y_test, model.predict(X_test))):.4f}')

# 出于速度,只算测试集前 1000 个样本的 SHAP
explainer  = shap.TreeExplainer(model)
shap_values = explainer(X_test.iloc[:1000])
print(f'\nshap_values 形状: {shap_values.values.shape}'
      f'  (即 1000 样本 × 8 特征,每个样本每特征一个推力值)')


### 6.4 单样本解释:Waterfall 图

挑测试集中第 0 行(具体某个街区),看看模型对它的预测是怎么「凑」出来的。


In [ ]:
# 选一个具体样本
idx = 0
print('该样本的特征值:')
print(X_test.iloc[idx].round(3))
print(f'\n真实房价:  {y_test.iloc[idx]:.3f}')
print(f'模型预测:  {model.predict(X_test.iloc[[idx]])[0]:.3f}')
expected_baseline = float(np.asarray(explainer.expected_value).flatten()[0])
print(f'训练集均值预测(基线): {expected_baseline:.3f}')

shap.plots.waterfall(shap_values[idx], show=False)
plt.tight_layout()
plt.show()


**怎么读这张图?**
- 底部 **E[f(x)]** = 训练集平均预测值(我们的「基线」);
- 顶部 **f(x)** = 模型对这个样本的实际预测值;
- 中间每一条 = 一个特征的 SHAP 值;**红色推高、蓝色压低**;
- 条越长,这个特征对最终预测的贡献越大;
- **从底部到顶部所有箭头叠加 = 预测值**(这是数学保证)。

这样你就能跟客户/老板清晰解释:「这个街区被预测高,**主要因为收入高(+X)和靠海(+Y)**;
但人均房间数偏少(-Z)拉低了一点。」


### 6.5 全局解释:Summary 图

把所有样本的 SHAP 值堆在一起,可以同时看到「**重要性 + 方向**」。


In [ ]:
shap.plots.beeswarm(shap_values, show=False)
plt.tight_layout()
plt.show()


**怎么读这张图?**
- 每行一个特征,**从上到下按整体重要性排序**;
- 每个点 = 一个样本;
- **横坐标 = SHAP 值**(大于 0 推高房价,小于 0 压低);
- **颜色 = 特征本身的值**(红色 = 该特征值高,蓝色 = 低)。

最有信息量的细节:**点的颜色与 SHAP 值的关系**告诉你**正/负相关性**:
- `MedInc`:红点(高收入)在右(推高),蓝点(低收入)在左(压低)→ **正相关**;
- `Latitude`:红点(纬度高,北加州内陆)在左,蓝点(纬度低,南加州沿海)在右 → 复杂的非线性关系。

这是用 `feature_importances_` **永远看不到**的信息。


### 6.6 单特征影响:Dependence 图

「`MedInc` 越高,房价就越高」——但具体是线性关系还是阈值关系?Dependence 图直接画出来。


In [ ]:
shap.plots.scatter(shap_values[:, 'MedInc'],
                   color=shap_values[:, 'AveOccup'], show=False)
plt.tight_layout()
plt.show()


**观察**:
- `MedInc` 与 SHAP 值近似单调上升,**但有明显拐点**——收入超过约 5(即 \$5 万)后拉抬效应开始减弱;
- 颜色编码的 `AveOccup`(每户人数)显示了**特征交互**:同样收入下,户均人数少的街区房价被推得更高。

> 📘 **小知识:SHAP 还能做什么?**
> - **公平性审计**:某种族/性别的 SHAP 分布是否被某些特征系统性压低?
> - **模型调试**:某个特征 SHAP 异常时,可能数据出了问题或泄漏;
> - **业务决策**:房产中介可以用 SHAP 告诉房主「装修能给你的房子加 \$20K」。

> ⚠️ **SHAP 的边界**:它解释的是**模型**,不是**真实世界**。模型如果学错了,SHAP 也会一本正经地解释错误。


## 7. 进一步学习导读

到这里,你已经掌握了表格回归的**基础流程 + 工程化 + 可解释性**全套打法。再往后,根据兴趣分四个方向。

### 7.1 想打牢理论基础

📕 **《An Introduction to Statistical Learning》**(ISLR,免费 PDF [statlearning.com](https://www.statlearning.com))

| 章 | 内容 | 与本系列对应 |
| --- | --- | --- |
| 第 3 章 | 线性回归(普通最小二乘) | 我们的 `LinearRegression` 基线 |
| 第 6 章 | 子集选择 + 收缩(Ridge/Lasso) | 我们的「正则化」一节 |
| 第 8 章 | 决策树、Bagging、随机森林、Boosting | 我们的「树模型」与进阶教程 |

ISLR 的特点:**写给统计/计算机本科生**,数学最低限度,大量直觉与图。中文有翻译版。
读完 3、6、8 章你就能比较 confidently 地参与机器学习讨论。

### 7.2 想再榨一轮性能

| 工具 | 一句话介绍 |
| --- | --- |
| **Optuna** | 贝叶斯超参搜索,比 GridSearch 智能、比 RandomSearch 高效 |
| **Stacking** | 把 XGBoost / LightGBM / CatBoost 的预测加权融合,通常再涨 1-2% |
| **特征选择** | `RFECV`、Boruta、SHAP-based 选择,把无用特征剔除 |
| **目标编码** | 类别特征不做独热而是用「该类别的目标均值」编码 |

### 7.3 想了解前沿:表格数据上的深度学习

业界至今争论不休:**DL 能在表格上打赢 GBM 吗?** 目前主流认知:
- **数据量百万以下** → GBM 仍然占优,DL 几乎没必要;
- **数据量百万以上 + 复杂特征交互** → 部分 DL 架构可以追平甚至略胜。

| 模型 | 出处 | 思路 |
| --- | --- | --- |
| **TabNet** | Google 2019 | 用注意力机制做「软特征选择」,可解释性较好 |
| **FT-Transformer** | Yandex 2021 | 把表格列作为 Transformer 的 token,效果最稳 |
| **SAINT** | 2021 | 在样本之间做 self-attention,能利用样本相似性 |

> ⚠️ **务实警告**:这些模型**调参极其敏感**,默认参数下经常输给 LightGBM 默认参数。
> 如果你在工业项目里,优先 LightGBM。如果你在做研究或参加 Kaggle 顶级比赛,可以试试。

### 7.4 想找更新更大的数据

加州数据是 1990 年的,信号有限。想做真实房价预测可以看:

| 数据源 | 特点 |
| --- | --- |
| **[Inside Airbnb](http://insideairbnb.com)** | 全球城市 Airbnb 房源(每月更新),含价格、位置、设施、评论 |
| **[Zillow Research](https://www.zillow.com/research/data/)** | 美国房价指数与单户成交数据 |
| **[Kaggle House Prices: Advanced Regression](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)** | 79 个特征的 Ames Housing 经典竞赛 |
| **[Kaggle Tabular Playground Series](https://www.kaggle.com/competitions?search=tabular)** | 每月一个新表格回归/分类竞赛 |
| **[OpenML](https://www.openml.org)** | 数千个公开数据集,一行 `fetch_openml` 加载 |

练习建议:**找一个 Inside Airbnb 城市数据,用本系列学到的全部技巧做一个端到端项目**——
这是最快从「跟着教程跑」过渡到「自己做项目」的方式。


## 8. 总结

本 notebook 教了 5 个工程/可解释性工具,以及 4 个延伸学习方向:

| 工具 | 一句话记忆 |
| --- | --- |
| `Pipeline` | **预处理 + 模型 = 一个对象**,自动杜绝数据泄漏 |
| `ColumnTransformer` | **不同列不同处理**,数值标准化 + 类别独热一气呵成 |
| `cross_validate` | **一次拿多个指标 + 训练分数**,直接看出过拟合 |
| `HistGradientBoostingRegressor` | **替代 GradientBoostingRegressor 的默认选择**,快 5–20× |
| **SHAP** | **解释每一个具体预测**,同时给出全局重要性 + 方向 + 交互 |

### 写在最后:你已经走了多远?

回头看一下你已经做到的事:
- ✅ 用线性、正则化、树、神经网络都做过加州房价预测
- ✅ 用 EDA / 残差 / 特征重要性 诊断过模型
- ✅ 用 GridSearchCV 调过超参
- ✅ 用海岸线距离做过领域特征
- ✅ 用 XGBoost / LightGBM / CatBoost 做过模型对比
- ✅ 用 Pipeline / ColumnTransformer 写过工程级代码
- ✅ 用 SHAP 解释过具体样本

这就是一个**机器学习工程师入门级**的能力组合。
下一步**最重要的不是再学一个工具,而是找一个真实问题完整做一遍**——
推荐顺序:Kaggle 入门竞赛 → 找一个开源数据集做小项目 → 在工作场景里找一个简单回归/分类问题。

> 💪 **真正的成长**永远来自**做出一个属于自己的项目**,而不是看完更多教程。
